[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/D.%20Integrated%20Tactical%20%26%20Pre-Planning%20Problems%20%28Connecting%20the%20Silos%29/29.%20The%20Integrated%20Quay%20Crane%20%26%20Yard%20Truck%20Scheduling%20Problem/P29-Tier-6_executed.ipynb)

---


# 29. The Integrated Quay Crane & Yard Truck Scheduling Problem

## Tier 6: An Agent-Based Coordination Simulation

### Goal

Build a small simulation in which crane and truck agents estimate their ability to handle containers, submit bids, and compete in sequential auctions. The simulation also demonstrates agent beliefs, intentions, adaptive specialization scores, and random operational failures.

The implementation is an educational agent-based simulation, not a complete terminal scheduling optimizer. In particular, cranes and trucks are treated as interchangeable bidders for a container; their real-world roles are not linked into a crane-to-truck transport chain.

### Inputs and outputs

The inputs are container attributes, agent capabilities, and the random seed set in the setup cell. The simulation returns allocation, auction, and agent-performance records. Additional experiments examine performance under specified agent failures and changes in specialization scores.

### Model assumptions and scope

- Every container is auctioned sequentially. Each available agent submits one bid, and the highest bid wins.
- A bid is a heuristic score based on estimated handling time and reliability; it is not a formally derived economic valuation.
- The payment rule below is a simplified auction rule inspired by externality-based payments. It is not the general Vickrey-Clarke-Groves (VCG) mechanism and does not establish VCG truthfulness or efficiency guarantees.
- Operational success is sampled probabilistically. Failure rates are per 1,000 operations, while reliability is used as the success-probability baseline.
- Agent learning methods and message-passing interfaces are included as building blocks, but the vessel simulation does not train a Q-table, run consensus, form coalitions, or exchange messages over a distributed network.
- Container deadlines, vessel precedence constraints, crane interference, truck routing, and physical capacity constraints are outside this model.

### Core auction logic

For container \(c\), let \(B_c\) be the set of bids from agents that are not failed. If \(B_c\) is nonempty, the winner is

$$
w_c = \arg\max_{i \in B_c} b_{ic},
$$

where \(b_{ic}\) is agent \(i\)'s heuristic bid for container \(c\). The implementation sorts bids in descending order and selects the first. Ties are resolved by the stable order of the bids presented to the sort.

The winner's payment is calculated by the simplified rule

$$
p_c = b_{(2)c} + 0.1\max(0, b_{(1)c} - b_{(2)c}),
$$

where \(b_{(1)c}\) and \(b_{(2)c}\) are the highest and second-highest bids. If there is only one bidder, the second-highest bid is zero. This rule is a didactic heuristic, not a general VCG payment formula.

### How to interpret the experiments

The simulated outcomes depend on the chosen assumptions and random draws. They illustrate how this implementation allocates work and responds to sampled failures; they do not validate claims about throughput, resilience, or performance in a real terminal.

In [ ]:
from collections import defaultdict, deque
from dataclasses import dataclass, field
from enum import Enum
import queue
import random
import time
from typing import Dict, List, Optional, Set

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Consistent plot styling makes comparisons across the experiments easier to read.
plt.style.use("default")
sns.set_palette("husl")

# A fixed seed makes the stochastic examples reproducible when run in order.
np.random.seed(42)
random.seed(42)

In [ ]:
class AgentType(Enum):
    """Operational role assigned to an agent in this simulation."""

    CRANE = "crane"
    TRUCK = "truck"
    COORDINATOR = "coordinator"


class AgentState(Enum):
    """Operational state used to determine whether an agent may bid."""

    IDLE = "idle"
    BIDDING = "bidding"
    WORKING = "working"
    FAILED = "failed"
    RECOVERING = "recovering"


@dataclass
class Container:
    """Container attributes used by the bidding and specialization rules."""

    id: int
    weight: float  # tons
    yard_location: int  # yard block, numbered 1 through 4
    priority: float  # between 0 and 1; larger values indicate greater urgency
    handling_difficulty: float  # between 0 and 1
    deadline: Optional[float] = None  # minutes from the start; not enforced here


@dataclass
class Bid:
    """Heuristic bid and handling-time estimate submitted by one agent."""

    agent_id: str
    container_id: int
    bid_amount: float
    estimated_completion_time: float
    confidence: float
    coalition_partners: List[str] = field(default_factory=list)


@dataclass
class AuctionResult:
    """Outcome of one auction under the simplified payment rule."""

    container_id: int
    winner_agent: str
    winning_bid: float
    second_highest_bid: float
    payment_amount: float
    externality_cost: float
    coalition: List[str]


@dataclass
class AgentCapability:
    """Processing and reliability parameters associated with an agent."""

    processing_speed: float  # containers per hour
    reliability: float  # probability parameter between 0 and 1
    current_load: float  # utilization proxy between 0 and 1
    specialization_score: Dict[str, float]
    failure_rate: float  # failures per 1,000 operations


@dataclass
class AgentMessage:
    """Message structure for the agent communication interface."""

    sender: str
    receiver: str
    message_type: str
    content: Dict
    timestamp: float

In [ ]:
class AutonomousAgent:
    """Agent with belief, desire, and intention data plus heuristic behavior.

    The BDI labels organize the agent's information and goals. They do not, by
    themselves, constitute a complete BDI reasoning or distributed-learning
    algorithm.
    """

    def __init__(self, agent_id: str, agent_type: AgentType,
                 initial_capability: AgentCapability):
        self.agent_id = agent_id
        self.agent_type = agent_type
        self.state = AgentState.IDLE

        # Beliefs are the information this agent currently stores about the system.
        self.beliefs = {
            "current_time": 0.0,
            "available_containers": [],
            "other_agents": {},
            "system_load": 0.0,
            "market_conditions": {},
        }
        self.desires = {
            "maximize_utilization": 0.7,
            "minimize_costs": 0.3,
            "maintain_reliability": 0.8,
            "learn_and_improve": 0.5,
        }
        self.intentions = []

        self.capability = initial_capability
        self.current_performance = {
            "containers_processed": 0,
            "total_revenue": 0.0,
            "total_costs": 0.0,
            "successful_operations": 0,
            "failed_operations": 0,
            "average_completion_time": 0.0,
        }

        # The Q-table methods are available for instruction, but are not called
        # by the sequential auction simulation in this notebook.
        self.q_table = defaultdict(lambda: defaultdict(float))
        self.learning_rate = 0.1
        self.discount_factor = 0.9
        self.exploration_rate = 0.1

        self.message_queue = queue.Queue()
        self.neighbors: Set[str] = set()
        self.specialization_history = deque(maxlen=100)

    def update_beliefs(self, new_information: Dict) -> None:
        """Replace recognized belief entries with newly supplied information."""
        for key, value in new_information.items():
            if key in self.beliefs:
                self.beliefs[key] = value

    def calculate_bid(self, container: Container) -> Bid:
        """Estimate handling time and convert it into a heuristic bid."""
        base_time = 60.0 / self.capability.processing_speed
        difficulty_multiplier = 1.0 + container.handling_difficulty * 0.5
        weight_multiplier = 1.0 + (container.weight - 20.0) / 40.0
        load_multiplier = 1.0 + self.capability.current_load * 0.3

        container_type = self._classify_container(container)
        specialization_bonus = self.capability.specialization_score.get(
            container_type, 0.0
        )
        specialization_multiplier = 1.0 - specialization_bonus * 0.2

        estimated_time = (
            base_time
            * difficulty_multiplier
            * weight_multiplier
            * load_multiplier
            * specialization_multiplier
        )

        # The score rewards short estimated times and reliable agents. It is a
        # ranking heuristic, not a monetary valuation derived from utility.
        base_value = 100.0 / max(estimated_time, 1.0)
        bid_amount = base_value * self.capability.reliability

        if self.beliefs["system_load"] > 0.8:
            bid_amount *= 1.2
        elif self.beliefs["system_load"] < 0.4:
            bid_amount *= 0.9

        confidence = min(
            1.0,
            self.capability.reliability
            * (1.0 - self.capability.failure_rate / 100.0),
        )

        return Bid(
            agent_id=self.agent_id,
            container_id=container.id,
            bid_amount=bid_amount,
            estimated_completion_time=estimated_time,
            confidence=confidence,
        )

    def _classify_container(self, container: Container) -> str:
        """Map container attributes to the categories used for specialization."""
        if container.weight > 30:
            return "heavy"
        if container.priority > 0.8:
            return "urgent"
        if container.handling_difficulty > 0.7:
            return "difficult"
        if container.yard_location in [3, 4]:
            return "far_yard"
        return "standard"

    def learn_from_experience(self, state: str, action: str, reward: float,
                              next_state: str) -> None:
        """Apply one tabular Q-learning update to a stored state-action value.

        The update moves the current estimate toward the observed reward plus
        the discounted best value available in the next state.
        """
        old_value = self.q_table[state][action]
        next_values = self.q_table[next_state].values()
        next_max = max(next_values) if next_values else 0.0
        target = reward + self.discount_factor * next_max
        self.q_table[state][action] = old_value + self.learning_rate * (
            target - old_value
        )

    def select_action(self, state: str, available_actions: List[str]) -> str:
        """Choose an action using an epsilon-greedy policy."""
        if not available_actions:
            raise ValueError("available_actions must contain at least one action")

        if random.random() < self.exploration_rate:
            return random.choice(available_actions)

        q_values = self.q_table[state]
        best_value = max(q_values[action] for action in available_actions)
        best_actions = [
            action for action in available_actions
            if q_values[action] == best_value
        ]
        return random.choice(best_actions)

    def update_performance(self, container_processed: bool,
                           completion_time: float, revenue: float,
                           cost: float) -> None:
        """Accumulate operational and financial metrics for this agent."""
        metrics = self.current_performance
        metrics["containers_processed"] += 1
        metrics["total_revenue"] += revenue
        metrics["total_costs"] += cost

        if container_processed:
            metrics["successful_operations"] += 1
        else:
            metrics["failed_operations"] += 1

        total_operations = (
            metrics["successful_operations"] + metrics["failed_operations"]
        )
        previous_average = metrics["average_completion_time"]
        metrics["average_completion_time"] = (
            previous_average * (total_operations - 1) + completion_time
        ) / total_operations

    def update_specialization(self, container: Container, success: bool) -> None:
        """Adjust the observed category score after a success or failure."""
        container_type = self._classify_container(container)
        if container_type not in self.capability.specialization_score:
            self.capability.specialization_score[container_type] = 0.5

        change = 0.05 if success else -0.02
        score = self.capability.specialization_score[container_type] + change
        self.capability.specialization_score[container_type] = max(
            0.0, min(1.0, score)
        )
        self.specialization_history.append({
            "container_type": container_type,
            "specialization_score": self.capability.specialization_score[container_type],
            "success": success,
        })

    def send_message(self, receiver: str, message_type: str,
                     content: Dict) -> AgentMessage:
        """Create a message record for a coordinator-managed simulation."""
        return AgentMessage(
            sender=self.agent_id,
            receiver=receiver,
            message_type=message_type,
            content=content,
            timestamp=self.beliefs["current_time"],
        )

    def receive_message(self, message: AgentMessage):
        """Queue and handle supported message types."""
        self.message_queue.put(message)

        if message.message_type == "update_beliefs":
            self.update_beliefs(message.content)
        elif message.message_type == "request_bid":
            container = message.content["container"]
            bid = self.calculate_bid(container)
            return self.send_message(
                message.sender, "submit_bid", {"bid": bid}
            )
        elif message.message_type == "auction_result":
            result = message.content["result"]
            if result.winner_agent == self.agent_id:
                self.intentions.append(
                    ("process_container", result.container_id)
                )
        return None

In [ ]:
class VCGAuctionMechanism:
    """Run highest-bid auctions with a simplified externality-style payment.

    The class name preserves the example's auction topic. The implemented
    payment is deliberately identified as a heuristic; it is not a general VCG
    mechanism and has no claimed VCG incentive or efficiency guarantee.
    """

    def __init__(self):
        self.auction_history = []
        self.market_prices = defaultdict(list)

    def run_auction(self, container: Container,
                    bidders: List[AutonomousAgent],
                    current_time: float) -> Optional[AuctionResult]:
        """Collect bids, allocate to the highest bidder, and record the result."""
        bids = [
            bidder.calculate_bid(container)
            for bidder in bidders
            if bidder.state != AgentState.FAILED
        ]
        if not bids:
            return None

        bids.sort(key=lambda bid: bid.bid_amount, reverse=True)
        winning_bid = bids[0]
        second_highest_bid = bids[1] if len(bids) > 1 else None
        second_amount = (
            second_highest_bid.bid_amount if second_highest_bid else 0.0
        )

        externality_cost = self._calculate_externality_cost(
            winning_bid, bids[1:]
        )
        payment_amount = second_amount + externality_cost

        result = AuctionResult(
            container_id=container.id,
            winner_agent=winning_bid.agent_id,
            winning_bid=winning_bid.bid_amount,
            second_highest_bid=second_amount,
            payment_amount=payment_amount,
            externality_cost=externality_cost,
            coalition=winning_bid.coalition_partners,
        )

        container_type = self._classify_container_for_market(container)
        self.market_prices[container_type].append(winning_bid.bid_amount)
        self.auction_history.append({
            "timestamp": current_time,
            "container_id": container.id,
            "winner": winning_bid.agent_id,
            "winning_bid": winning_bid.bid_amount,
            "num_bidders": len(bids),
            "payment": payment_amount,
        })
        return result

    def _calculate_externality_cost(self, winning_bid: Bid,
                                   other_bids: List[Bid]) -> float:
        """Return the example's 10% winner-to-runner-up gap adjustment."""
        if not other_bids:
            return 0.0
        next_best_bid = max(other_bids, key=lambda bid: bid.bid_amount)
        return max(
            0.0, winning_bid.bid_amount - next_best_bid.bid_amount
        ) * 0.1

    def _classify_container_for_market(self, container: Container) -> str:
        """Use the market-tracking categories defined by this example."""
        if container.weight > 30:
            return "heavy"
        if container.priority > 0.8:
            return "urgent"
        return "standard"

    def get_market_price_trend(self, container_type: str) -> Dict:
        """Summarize recent recorded winning bids for a container category."""
        prices = self.market_prices[container_type]
        if not prices:
            return {
                "trend": "insufficient_data",
                "avg_price": 0.0,
                "price_volatility": 0.0,
            }

        recent_prices = prices[-10:]
        recent_avg = float(np.mean(recent_prices))
        if len(prices) < 2:
            trend = "insufficient_data"
        else:
            older_prices = prices[-20:-10]
            if not older_prices:
                trend = "insufficient_data"
            else:
                older_avg = float(np.mean(older_prices))
                if recent_avg > older_avg * 1.05:
                    trend = "increasing"
                elif recent_avg < older_avg * 0.95:
                    trend = "decreasing"
                else:
                    trend = "stable"

        volatility = (
            float(np.std(recent_prices) / recent_avg)
            if recent_avg > 0
            else 0.0
        )
        return {
            "trend": trend,
            "avg_price": recent_avg,
            "price_volatility": volatility,
        }

In [ ]:
class MultiAgentSystem:
    """Coordinate sequential auctions among simulated crane and truck agents."""

    def __init__(self, num_cranes: int = 3, num_trucks: int = 4):
        if num_cranes < 0 or num_trucks < 0 or num_cranes + num_trucks == 0:
            raise ValueError("At least one crane or truck agent is required")

        self.num_cranes = num_cranes
        self.num_trucks = num_trucks
        self.agents = {}
        self.crane_agents = {}
        self.truck_agents = {}
        self.auction_mechanism = VCGAuctionMechanism()
        self.current_time = 0.0
        self.system_performance = {
            "total_containers_processed": 0,
            "total_revenue": 0.0,
            "total_costs": 0.0,
            "agent_failures": 0,
            "auctions_completed": 0,
            "coalitions_formed": 0,
        }
        self.global_performance_history = []
        self.specialization_patterns = defaultdict(list)
        self.failed_agents = set()
        self.recovery_attempts = {}

        self._initialize_agents()
        self._setup_agent_network()

    def _initialize_agents(self) -> None:
        """Create agents with the capability ranges used in the example."""
        for i in range(self.num_cranes):
            crane_id = f"QC{i + 1}"
            capability = AgentCapability(
                processing_speed=np.random.uniform(25, 35),
                reliability=np.random.uniform(0.85, 0.95),
                current_load=0.0,
                specialization_score={
                    "heavy": np.random.uniform(0.3, 0.7),
                    "urgent": np.random.uniform(0.4, 0.8),
                    "difficult": np.random.uniform(0.2, 0.6),
                    "far_yard": np.random.uniform(0.3, 0.7),
                    "standard": np.random.uniform(0.5, 0.9),
                },
                failure_rate=np.random.uniform(1, 5),
            )
            agent = AutonomousAgent(crane_id, AgentType.CRANE, capability)
            self.agents[crane_id] = agent
            self.crane_agents[crane_id] = agent

        for i in range(self.num_trucks):
            truck_id = f"T{i + 1}"
            capability = AgentCapability(
                processing_speed=np.random.uniform(15, 25),
                reliability=np.random.uniform(0.80, 0.90),
                current_load=0.0,
                specialization_score={
                    "heavy": np.random.uniform(0.2, 0.6),
                    "urgent": np.random.uniform(0.3, 0.7),
                    "difficult": np.random.uniform(0.4, 0.8),
                    "far_yard": np.random.uniform(0.5, 0.9),
                    "standard": np.random.uniform(0.4, 0.8),
                },
                failure_rate=np.random.uniform(2, 8),
            )
            agent = AutonomousAgent(truck_id, AgentType.TRUCK, capability)
            self.agents[truck_id] = agent
            self.truck_agents[truck_id] = agent

    def _setup_agent_network(self) -> None:
        """Record a fully connected neighbor set for the communication model."""
        for agent_id, agent in self.agents.items():
            agent.neighbors = set(self.agents) - {agent_id}

    def process_vessel_arrival(self, containers: List[Container]) -> Dict:
        """Auction containers in input order and record processing outcomes."""
        print(f"Processing vessel arrival with {len(containers)} containers")
        processing_results = {
            "containers_allocated": [],
            "auction_results": [],
            "agent_performance": {},
            "coalitions_formed": [],
            "failures": [],
            "processing_time": 0.0,
        }
        start_time = time.time()

        for container in containers:
            self._simulate_random_failures()
            available_agents = [
                agent for agent in self.agents.values()
                if agent.state != AgentState.FAILED
            ]
            if not available_agents:
                print("No available agents; processing stops.")
                break

            auction_result = self.auction_mechanism.run_auction(
                container, available_agents, self.current_time
            )
            if auction_result is not None:
                working_count = sum(
                    agent.state == AgentState.WORKING
                    for agent in self.agents.values()
                )
                system_load = working_count / len(self.agents)
                remaining_containers = containers[containers.index(container):]
                for agent in available_agents:
                    agent.update_beliefs({
                        "system_load": system_load,
                        "current_time": self.current_time,
                        "available_containers": remaining_containers,
                    })

                winner_agent = self.agents[auction_result.winner_agent]
                winner_agent.receive_message(AgentMessage(
                    sender="coordinator",
                    receiver=auction_result.winner_agent,
                    message_type="auction_result",
                    content={"result": auction_result},
                    timestamp=self.current_time,
                ))

                winning_bid = next(
                    bid for bid in available_agents
                    if bid.agent_id == auction_result.winner_agent
                ).calculate_bid(container)
                success = self._process_container_by_agent(
                    winner_agent, container, auction_result
                )
                completion_time = winning_bid.estimated_completion_time

                if success:
                    processing_results["containers_allocated"].append(container.id)
                    winner_agent.update_performance(
                        True, completion_time,
                        auction_result.payment_amount,
                        auction_result.payment_amount * 0.8,
                    )
                    winner_agent.update_specialization(container, True)
                    self.system_performance["total_containers_processed"] += 1
                    self.system_performance["total_revenue"] += (
                        auction_result.payment_amount
                    )
                    self.system_performance["total_costs"] += (
                        auction_result.payment_amount * 0.8
                    )
                else:
                    processing_results["failures"].append(container.id)
                    winner_agent.update_performance(
                        False, completion_time, 0.0,
                        auction_result.payment_amount * 0.5,
                    )
                    winner_agent.update_specialization(container, False)
                    self.system_performance["total_costs"] += (
                        auction_result.payment_amount * 0.5
                    )

                processing_results["auction_results"].append(auction_result)
                self.system_performance["auctions_completed"] += 1

            self.current_time += 1.0

        processing_results["processing_time"] = time.time() - start_time
        processing_results["agent_performance"] = {
            agent_id: agent.current_performance.copy()
            for agent_id, agent in self.agents.items()
        }
        return processing_results

    def _simulate_random_failures(self) -> None:
        """Sample new agent failures using the per-operation failure rates."""
        for agent_id, agent in self.agents.items():
            if agent.state == AgentState.FAILED:
                continue
            if random.random() < agent.capability.failure_rate / 1000.0:
                agent.state = AgentState.FAILED
                self.failed_agents.add(agent_id)
                self.system_performance["agent_failures"] += 1
                print(f"Agent {agent_id} failed.")

    def _process_container_by_agent(self, agent: AutonomousAgent,
                                    container: Container,
                                    auction_result: AuctionResult) -> bool:
        """Sample processing success from reliability, load, and specialization."""
        success_probability = agent.capability.reliability
        success_probability -= agent.capability.current_load * 0.1
        container_type = self._classify_container_for_agent(container)
        specialization_bonus = agent.capability.specialization_score.get(
            container_type, 0.0
        )
        success_probability += specialization_bonus * 0.1
        success_probability = max(0.1, min(1.0, success_probability))
        success = random.random() < success_probability

        if success:
            agent.capability.current_load = min(
                1.0, agent.capability.current_load + 0.1
            )
        else:
            agent.capability.current_load = max(
                0.0, agent.capability.current_load - 0.05
            )
        agent.capability.current_load *= 0.95
        return success

    def _classify_container_for_agent(self, container: Container) -> str:
        """Use the same category order as the agent bidding rule."""
        if container.weight > 30:
            return "heavy"
        if container.priority > 0.8:
            return "urgent"
        if container.handling_difficulty > 0.7:
            return "difficult"
        if container.yard_location in [3, 4]:
            return "far_yard"
        return "standard"

    def analyze_system_performance(self) -> Dict:
        """Summarize realized processing, agent state, and recorded bid trends."""
        auctions = self.system_performance["auctions_completed"]
        processed = self.system_performance["total_containers_processed"]
        analysis = {
            "system_efficiency": processed / auctions if auctions else 0.0,
            "agent_specialization": {},
            "learning_progress": {},
            "fault_tolerance": 0.0,
            "market_dynamics": {},
            "coordination_quality": 0.0,
        }

        for agent_id, agent in self.agents.items():
            scores = agent.capability.specialization_score
            if scores:
                primary = max(scores, key=scores.get)
                analysis["agent_specialization"][agent_id] = {
                    "primary_specialization": primary,
                    "specialization_level": scores[primary],
                    "specialization_diversity": sum(
                        score > 0.6 for score in scores.values()
                    ),
                }

            q_values = [
                value
                for state_actions in agent.q_table.values()
                for value in state_actions.values()
            ]
            analysis["learning_progress"][agent_id] = {
                "q_table_size": len(q_values),
                "avg_q_value": float(np.mean(q_values)) if q_values else 0.0,
                "exploration_rate": agent.exploration_rate,
            }

        total_agents = len(self.agents)
        operational_agents = sum(
            agent.state != AgentState.FAILED for agent in self.agents.values()
        )
        analysis["fault_tolerance"] = (
            operational_agents / total_agents if total_agents else 0.0
        )
        for container_type in ["heavy", "urgent", "standard"]:
            analysis["market_dynamics"][container_type] = (
                self.auction_mechanism.get_market_price_trend(container_type)
            )

        average_reliability = float(np.mean([
            agent.capability.reliability for agent in self.agents.values()
        ]))
        analysis["coordination_quality"] = (
            average_reliability * analysis["fault_tolerance"]
        )
        return analysis

In [ ]:
def create_vessel_scenario(num_containers: int = 240) -> List[Container]:
    """Generate containers using the example's stated distributions."""
    if num_containers < 0:
        raise ValueError("num_containers must be non-negative")

    containers = []
    for i in range(num_containers):
        weight = int(np.random.choice(
            [10, 20, 30, 40], p=[0.3, 0.4, 0.2, 0.1]
        ))
        priority = float(np.random.beta(2, 5))
        handling_difficulty = float(np.random.beta(1.5, 3))
        yard_location = int(np.random.choice(
            [1, 2, 3, 4], p=[0.4, 0.3, 0.2, 0.1]
        ))
        deadline = (
            float(np.random.uniform(30, 180)) if priority > 0.7 else None
        )
        containers.append(Container(
            id=i + 1,
            weight=weight,
            yard_location=yard_location,
            priority=priority,
            handling_difficulty=handling_difficulty,
            deadline=deadline,
        ))
    return containers


def create_multi_agent_system() -> MultiAgentSystem:
    """Create the example system with three cranes and four trucks."""
    mas = MultiAgentSystem(num_cranes=3, num_trucks=4)
    print("Multi-Agent System Initialized:")
    print(f"- Crane agents: {len(mas.crane_agents)}")
    print(f"- Truck agents: {len(mas.truck_agents)}")
    print(f"- Total agents: {len(mas.agents)}")
    print("\nInitial Agent Capabilities:")
    for agent_id, agent in list(mas.agents.items())[:3]:
        print(
            f"{agent_id}: Speed={agent.capability.processing_speed:.1f}/hr, "
            f"Reliability={agent.capability.reliability:.2f}"
        )
    return mas


mas = create_multi_agent_system()
vessel_containers = create_vessel_scenario(240)
print(f"\nVessel scenario created: {len(vessel_containers)} containers")
print(
    "Container categories by the displayed thresholds: "
    f"Heavy={sum(c.weight > 30 for c in vessel_containers)}, "
    f"Priority above 0.7={sum(c.priority > 0.7 for c in vessel_containers)}"
)

In [ ]:
def run_multi_agent_simulation(mas: MultiAgentSystem,
                                containers: List[Container]):
    """Run one vessel simulation and return its results and analysis."""
    print("=== MULTI-AGENT SIMULATION START ===")
    results = mas.process_vessel_arrival(containers)
    print("\n=== MULTI-AGENT SIMULATION COMPLETE ===")
    print(f"Processing time: {results['processing_time']:.2f} seconds")
    print(
        f"Containers successfully processed: "
        f"{len(results['containers_allocated'])}/{len(containers)}"
    )
    print(f"Auctions completed: {len(results['auction_results'])}")
    print(f"Processing failures: {len(results['failures'])}")

    analysis = mas.analyze_system_performance()
    print("\nSystem Performance:")
    print(f"Success per completed auction: {analysis['system_efficiency']:.2%}")
    print(f"Operational agent fraction: {analysis['fault_tolerance']:.2%}")
    print(
        "Coordination proxy (mean reliability × operational fraction): "
        f"{analysis['coordination_quality']:.2%}"
    )
    return results, analysis


simulation_results, performance_analysis = run_multi_agent_simulation(
    mas, vessel_containers
)

In [ ]:
def visualize_multi_agent_results(results: Dict, analysis: Dict,
                                   mas: MultiAgentSystem,
                                   containers: List[Container]) -> None:
    """Plot allocation, auction, specialization, and system summaries."""
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle(
        "Agent-Based Crane-Truck Coordination Simulation",
        fontsize=16,
        fontweight="bold",
    )

    agent_ids = list(results["agent_performance"])
    processed = [
        results["agent_performance"][agent_id]["successful_operations"]
        for agent_id in agent_ids
    ]
    agent_colors = ["red" if agent_id.startswith("QC") else "blue"
                    for agent_id in agent_ids]
    bars = axes[0, 0].bar(agent_ids, processed, color=agent_colors, alpha=0.7)
    axes[0, 0].set_xlabel("Agents")
    axes[0, 0].set_ylabel("Successfully Processed Containers")
    axes[0, 0].set_title("Agent Processing Results")
    axes[0, 0].tick_params(axis="x", rotation=45)
    axes[0, 0].grid(True, alpha=0.3)
    for bar, count in zip(bars, processed):
        axes[0, 0].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            str(count),
            ha="center",
            va="bottom",
        )

    auction_results = results["auction_results"]
    winning_bids = [result.winning_bid for result in auction_results]
    payments = [result.payment_amount for result in auction_results]
    axes[0, 1].scatter(winning_bids, payments, alpha=0.6, s=30)
    axes[0, 1].set_xlabel("Winning Heuristic Bid (score units)")
    axes[0, 1].set_ylabel("Payment (score units)")
    axes[0, 1].set_title("Winning Bids and Simplified Payments")
    axes[0, 1].grid(True, alpha=0.3)
    if winning_bids:
        lower = min(min(winning_bids), min(payments))
        upper = max(max(winning_bids), max(payments))
        axes[0, 1].plot(
            [lower, upper], [lower, upper], "r--", alpha=0.5,
            label="Bid equals payment",
        )
        axes[0, 1].legend()

    specialization = analysis["agent_specialization"]
    spec_ids = list(specialization)
    spec_labels = [
        f"{agent_id}\n({specialization[agent_id]['primary_specialization']})"
        for agent_id in spec_ids
    ]
    spec_levels = [
        specialization[agent_id]["specialization_level"]
        for agent_id in spec_ids
    ]
    spec_colors = ["red" if agent_id.startswith("QC") else "blue"
                   for agent_id in spec_ids]
    axes[0, 2].bar(range(len(spec_labels)), spec_levels, color=spec_colors,
                   alpha=0.7)
    axes[0, 2].set_xlabel("Agent and Highest-Scoring Category")
    axes[0, 2].set_ylabel("Specialization Score")
    axes[0, 2].set_title("Current Specialization Scores")
    axes[0, 2].set_xticks(range(len(spec_labels)))
    axes[0, 2].set_xticklabels(spec_labels, rotation=45, ha="right", fontsize=8)
    axes[0, 2].set_ylim(0, 1)
    axes[0, 2].grid(True, alpha=0.3)

    market = analysis["market_dynamics"]
    categories = list(market)
    prices = [market[category]["avg_price"] for category in categories]
    volatility = [market[category]["price_volatility"] for category in categories]
    axes[1, 0].bar(categories, prices, alpha=0.7, color="green")
    axes[1, 0].set_xlabel("Container Category")
    axes[1, 0].set_ylabel("Mean Recorded Winning Bid (score units)")
    axes[1, 0].set_title("Recorded Market Prices")
    axes[1, 0].grid(True, alpha=0.3)
    for index, (price, value) in enumerate(zip(prices, volatility)):
        axes[1, 0].text(index, price, f"CV={value:.2f}", ha="center", va="bottom")

    learning = analysis["learning_progress"]
    learning_ids = list(learning)
    q_sizes = [learning[agent_id]["q_table_size"] for agent_id in learning_ids]
    learning_colors = ["red" if agent_id.startswith("QC") else "blue"
                       for agent_id in learning_ids]
    axes[1, 1].bar(learning_ids, q_sizes, color=learning_colors, alpha=0.7)
    axes[1, 1].set_xlabel("Agents")
    axes[1, 1].set_ylabel("Stored State-Action Entries")
    axes[1, 1].set_title("Q-Table Entries (Not Trained in This Run)")
    axes[1, 1].tick_params(axis="x", rotation=45)
    axes[1, 1].grid(True, alpha=0.3)

    axes[1, 2].axis("off")
    revenue = mas.system_performance["total_revenue"]
    costs = mas.system_performance["total_costs"]
    profit = revenue - costs
    success_rate = (
        len(results["containers_allocated"]) / len(containers)
        if containers else 0.0
    )
    summary = (
        "SIMULATION SUMMARY\n"
        "==================\n\n"
        f"Containers in input: {len(containers)}\n"
        f"Successfully processed: {len(results['containers_allocated'])}\n"
        f"Processing success fraction: {success_rate:.1%}\n"
        f"Processing time: {results['processing_time']:.2f} seconds\n\n"
        "Accounting under the illustrative payment rule:\n"
        f"Recorded revenue: {revenue:,.2f} score units\n"
        f"Recorded costs: {costs:,.2f} score units\n"
        f"Net amount: {profit:,.2f} score units\n\n"
        "System indicators:\n"
        f"Success per completed auction: {analysis['system_efficiency']:.1%}\n"
        f"Operational agent fraction: {analysis['fault_tolerance']:.1%}\n"
        f"Coordination proxy: {analysis['coordination_quality']:.1%}\n"
        f"Agent failures: {mas.system_performance['agent_failures']}\n\n"
        "Included components:\n"
        "- Sequential highest-bid allocation\n"
        "- Probabilistic processing outcomes\n"
        "- Adaptive category scores\n"
        "- Agent message and Q-learning interfaces"
    )
    axes[1, 2].text(
        0.04, 0.96, summary, transform=axes[1, 2].transAxes,
        fontsize=9, verticalalignment="top", fontfamily="monospace",
        bbox={"boxstyle": "round", "facecolor": "lightyellow", "alpha": 0.8},
    )
    plt.tight_layout()
    plt.show()


visualize_multi_agent_results(
    simulation_results, performance_analysis, mas, vessel_containers
)

In [ ]:
def test_fault_tolerance():
    """Compare processing outcomes under controlled initial agent failures."""
    print("\n=== FAULT-TOLERANCE EXPERIMENT ===")
    failure_scenarios = [
        {"failed_agents": 0, "description": "No initial failures"},
        {"failed_agents": 1, "description": "One initial failure"},
        {"failed_agents": 2, "description": "Two initial failures"},
        {"failed_agents": 3, "description": "Three initial failures"},
    ]
    fault_tolerance_results = []

    for scenario in failure_scenarios:
        print(f"\nTesting: {scenario['description']}")
        test_mas = MultiAgentSystem(num_cranes=3, num_trucks=4)
        failed_count = min(scenario["failed_agents"], len(test_mas.agents))
        if failed_count:
            agents_to_fail = random.sample(
                list(test_mas.agents), failed_count
            )
            for agent_id in agents_to_fail:
                test_mas.agents[agent_id].state = AgentState.FAILED
                test_mas.failed_agents.add(agent_id)
            print(f"  Initially failed agents: {agents_to_fail}")

        test_containers = create_vessel_scenario(50)
        test_results = test_mas.process_vessel_arrival(test_containers)
        test_analysis = test_mas.analyze_system_performance()
        actual_throughput = len(test_results["containers_allocated"])
        throughput_maintenance = actual_throughput / len(test_containers)
        result = {
            "scenario": scenario["description"],
            "failed_agents": failed_count,
            "operational_agents": len(test_mas.agents) - failed_count,
            "containers_processed": actual_throughput,
            "throughput_maintenance": throughput_maintenance,
            "system_efficiency": test_analysis["system_efficiency"],
            "fault_tolerance_score": test_analysis["fault_tolerance"],
        }
        fault_tolerance_results.append(result)
        print(f"  Containers processed: {actual_throughput}/{len(test_containers)}")
        print(f"  Throughput fraction: {throughput_maintenance:.1%}")
        print(f"  Success per completed auction: {test_analysis['system_efficiency']:.1%}")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    scenario_labels = [result["scenario"] for result in fault_tolerance_results]
    throughput = [
        result["throughput_maintenance"] for result in fault_tolerance_results
    ]
    colors = [
        "green" if value > 0.8 else "orange" if value > 0.6 else "red"
        for value in throughput
    ]
    bars = ax1.bar(
        scenario_labels, [value * 100 for value in throughput],
        color=colors, alpha=0.7,
    )
    ax1.set_ylabel("Successfully Processed Containers (%)")
    ax1.set_title("Processing Fraction by Initial Failure Scenario")
    ax1.tick_params(axis="x", rotation=35)
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(0, 100)
    for threshold, label, color in [
        (80, "80% reference", "green"), (60, "60% reference", "orange")
    ]:
        ax1.axhline(threshold, color=color, linestyle="--", alpha=0.5,
                    label=label)
    ax1.legend()
    for bar, value in zip(bars, throughput):
        ax1.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{value * 100:.1f}%",
            ha="center",
            va="bottom",
        )

    failures = [result["failed_agents"] for result in fault_tolerance_results]
    efficiencies = [
        result["system_efficiency"] for result in fault_tolerance_results
    ]
    ax2.plot(failures, [value * 100 for value in efficiencies], "o-",
             color="blue", linewidth=2, markersize=8)
    ax2.set_xlabel("Initially Failed Agents")
    ax2.set_ylabel("Success per Completed Auction (%)")
    ax2.set_title("Processing Success and Initial Failures")
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim(0, 100)
    for failure_count, efficiency in zip(failures, efficiencies):
        ax2.annotate(
            f"{efficiency * 100:.1f}%",
            (failure_count, efficiency * 100),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=8,
        )

    plt.tight_layout()
    plt.show()
    return fault_tolerance_results


fault_tolerance_results = test_fault_tolerance()

In [ ]:
def _calculate_specialization_entropy(agent: AutonomousAgent) -> float:
    """Calculate Shannon entropy of normalized nonnegative category scores.

    Lower entropy indicates that the score distribution is more concentrated;
    it does not by itself prove that the agent performs better.
    """
    scores = [
        max(0.0, score)
        for score in agent.capability.specialization_score.values()
    ]
    total = sum(scores)
    if total == 0:
        return 0.0
    probabilities = [score / total for score in scores if score > 0]
    return float(-sum(p * np.log2(p) for p in probabilities))


def analyze_emergent_specialization():
    """Track category scores over five sequential vessel-arrival simulations.

    A fresh system isolates this experiment from the main simulation and the
    failure scenarios. The scores change only when an agent wins an auction and
    processing succeeds or fails; this is adaptive score updating, not Q-learning.
    """
    print("\n=== SPECIALIZATION SCORE ANALYSIS ===")
    experiment_mas = create_multi_agent_system()
    specialization_evolution = []

    for vessel_number in range(5):
        print(f"\nProcessing vessel {vessel_number + 1}...")
        containers = create_vessel_scenario(48)
        results = experiment_mas.process_vessel_arrival(containers)
        vessel_specialization = {}
        for agent_id, agent in experiment_mas.agents.items():
            scores = agent.capability.specialization_score
            primary = max(scores, key=scores.get)
            vessel_specialization[agent_id] = {
                "primary_specialization": primary,
                "max_score": scores[primary],
                "specialization_entropy": _calculate_specialization_entropy(agent),
            }
        specialization_evolution.append({
            "vessel_number": vessel_number + 1,
            "specializations": vessel_specialization,
            "containers_processed": len(results["containers_allocated"]),
        })

    print("\nSpecialization Score Summary:")
    for agent_id in experiment_mas.agents:
        records = [
            vessel["specializations"][agent_id]
            for vessel in specialization_evolution
        ]
        primary_categories = [
            record["primary_specialization"] for record in records
        ]
        most_common = max(set(primary_categories), key=primary_categories.count)
        consistency = primary_categories.count(most_common) / len(primary_categories)
        initial_score = records[0]["max_score"]
        final_score = records[-1]["max_score"]
        relative_change = (
            (final_score - initial_score) / initial_score
            if initial_score > 0 else 0.0
        )
        print(
            f"{agent_id}: Most frequent category={most_common} "
            f"({consistency:.1%} of observations), "
            f"relative change in highest score={relative_change:+.1%}"
        )

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    for agent_id in list(experiment_mas.agents)[:3]:
        vessels = [entry["vessel_number"] for entry in specialization_evolution]
        scores = [
            entry["specializations"][agent_id]["max_score"]
            for entry in specialization_evolution
        ]
        ax1.plot(vessels, scores, "o-", label=agent_id,
                 linewidth=2, markersize=6)
    ax1.set_xlabel("Vessel Arrival Number")
    ax1.set_ylabel("Highest Category Specialization Score")
    ax1.set_title("Specialization Scores Across Arrivals")
    ax1.legend(title="Agent")
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(0, 1)

    final_categories = defaultdict(int)
    for agent in experiment_mas.agents.values():
        primary = max(
            agent.capability.specialization_score,
            key=agent.capability.specialization_score.get,
        )
        final_categories[primary] += 1
    if final_categories:
        ax2.pie(
            list(final_categories.values()),
            labels=list(final_categories),
            autopct="%1.1f%%",
            startangle=90,
        )
    ax2.set_title("Highest-Scoring Category at Experiment End")
    plt.tight_layout()
    plt.show()
    return specialization_evolution


specialization_evolution = analyze_emergent_specialization()

## Interpretation, strengths, and limitations

### What the simulation demonstrates

- **Agent-level decisions:** Each agent calculates its own heuristic bid from capability, estimated handling time, reliability, load, and specialization score.
- **Auction-based allocation:** The coordinator collects bids and assigns each container to the highest bidder.
- **Adaptive category scores:** An agent's score for a container category rises after success and falls after failure, within the specified bounds.
- **Stochastic failures:** Agents can become unavailable, and processing outcomes are sampled probabilistically.

These mechanisms are implemented in a single Python process. The fully connected neighbor sets and message interface describe possible communication structures, but the experiments do not execute a distributed protocol or consensus algorithm. The Q-learning interface is likewise not used to update the agents during the experiments.

### Strengths and limitations

**Useful features of this teaching model:**

- The agent, bid, auction, and performance records are explicit and inspectable.
- Controlled failure scenarios make the effect of initial agent availability observable.
- The fixed random seed supports reproducible execution when cells are run in order.
- The specialization experiment separates score adaptation from the main simulation.

**Important limitations:**

- The bid is a heuristic score, and the payment rule is not a general VCG mechanism.
- Crane and truck roles are not coupled through a physical handling or transport plan.
- Sequential auctions do not optimize a joint schedule over time.
- Failure, reliability, load, and handling-time relationships are stylized assumptions.
- Deadlines, routing, interference, capacity, recovery, and actual message delays are not modeled.
- A single run is not statistical evidence of expected performance; repeated trials and uncertainty analysis would be needed for that purpose.

### Scope for further study

Natural extensions within this agent-based setting include modeling crane and truck compatibility, adding explicit recovery behavior, and evaluating repeated independent runs. Those additions would require new assumptions and are not part of the present simulation.